# Lab | Data Aggregation and Filtering - Solution Notebook

**Dataset**: Marketing Customer Analysis

**Source**: https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

## Setup - Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv")

# Quick look at the data
print("Shape:", df.shape)
df.head()

In [ ]:
# Clean column names — lowercase and replace spaces with underscores
df.columns = [col.lower().replace(' ', '_') for col in df.columns]
print("Columns:", df.columns.tolist())

---
## Exercise 1: Filter Customers with Low Claims and Marketing Response = Yes

**Goal**: Create a new DataFrame that only includes customers who:
- Have a total_claim_amount **below $1,000**
- Responded **"Yes"** to the last marketing campaign

In [ ]:
# Filter using two conditions combined with & (AND operator)
# Condition 1: total_claim_amount < 1000
# Condition 2: response == 'Yes'
# Both must be True for a row to be included

filtered_df = df[
    (df['total_claim_amount'] < 1000) &
    (df['response'] == 'Yes')
].reset_index(drop=True)

print(f"Original dataset: {len(df)} rows")
print(f"Filtered dataset: {len(filtered_df)} rows")
print()
filtered_df.head()

**Insight**: After filtering we have customers who both responded positively to marketing AND have low claim amounts — these are the most cost-effective customers for the company.

---
## Exercise 2: Average Monthly Premium and CLV by Policy Type and Gender (Yes Responders)

**Goal**: Analyze average monthly premium and CLV for customers who responded "Yes", grouped by policy type and gender.

In [ ]:
# Step 1: Filter only 'Yes' responders from the ORIGINAL DataFrame
yes_responders = df[df['response'] == 'Yes']

print(f"Number of 'Yes' responders: {len(yes_responders)}")

In [ ]:
# Step 2: Group by policy_type and gender, calculate average monthly premium and CLV
# groupby() groups the data by the specified columns
# .agg() lets us calculate multiple aggregations at once
analysis = yes_responders.groupby(['policy_type', 'gender']).agg(
    avg_monthly_premium=('monthly_premium_auto', 'mean'),
    avg_clv=('customer_lifetime_value', 'mean'),
    avg_total_claim=('total_claim_amount', 'mean')
).round(2)

print("Average Monthly Premium, CLV, and Total Claim Amount by Policy Type and Gender:")
print(analysis)

**Insights**:
- Compare avg_monthly_premium (revenue) vs avg_total_claim (cost) for each segment
- Segments where premium is high and claims are low are the most profitable
- Segments where claims are high relative to premium are high-risk for the company
- CLV gives a long-term profitability view — higher CLV segments are more valuable to retain

---
## Exercise 3: Total Customers by State (States with More Than 500 Customers)

**Goal**: Count customers per state and keep only states with more than 500.

In [ ]:
# Step 1: Count customers per state using groupby and size()
# .size() counts the number of rows in each group
state_counts = df.groupby('state').size()

print("All states and their customer counts:")
print(state_counts.sort_values(ascending=False))
print()

In [ ]:
# Step 2: Filter states with more than 500 customers
# Boolean indexing on a Series works the same way as on a DataFrame
states_over_500 = state_counts[state_counts > 500]

print("States with more than 500 customers:")
print(states_over_500.sort_values(ascending=False))

**Insight**: Only the states shown above have more than 500 customers. These are the key markets for the insurance company and should receive the most marketing investment.

---
## Exercise 4: Max, Min, and Median CLV by Education Level and Gender

**Goal**: Find the maximum, minimum, and median customer lifetime value grouped by education and gender.

In [ ]:
# Group by education and gender, then calculate max, min, and median for CLV
# .agg() with a list of functions calculates all of them at once
clv_analysis = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(
    ['max', 'min', 'median']
).round(2)

print("Max, Min, and Median CLV by Education Level and Gender:")
print(clv_analysis)

**Conclusions**:
- The max values show the highest potential value customers in each segment
- The median gives a more reliable central tendency than the mean (less affected by outliers)
- Compare across education levels to see if higher education correlates with higher CLV
- Compare across genders to identify any significant differences in customer value

---
## Bonus Exercise 5: Policies Sold by State and Month (Pivot Table)

**Goal**: Create a pivot table with states as rows and months as columns.

In [ ]:
# First extract the month from the effective_to_date column
# pd.to_datetime() converts the string date to a proper datetime object
# .dt.month extracts just the month number
df['month'] = pd.to_datetime(df['effective_to_date']).dt.month

# Create pivot table
# values = what to count/aggregate
# index = rows
# columns = columns
# aggfunc = how to aggregate (count here)
policies_pivot = df.pivot_table(
    values='policy',
    index='state',
    columns='month',
    aggfunc='count',
    fill_value=0  # Fill missing combinations with 0
)

print("Number of policies sold by state and month:")
policies_pivot

---
## Bonus Exercise 6: Top 3 States by Policies Sold - Monthly Breakdown

In [ ]:
# Step 1: Find total policies per state
total_by_state = df.groupby('state')['policy'].count().sort_values(ascending=False)

# Step 2: Get top 3 state names
top_3_states = total_by_state.head(3).index.tolist()
print("Top 3 states:", top_3_states)
print()

In [ ]:
# Step 3: Filter the pivot table to only show top 3 states
top_3_pivot = policies_pivot.loc[top_3_states]

print("Policies sold by month for top 3 states:")
top_3_pivot

---
## Bonus Exercise 7: Marketing Channel Response Rate using Melt

In [ ]:
# Step 1: Create a pivot table of response counts by sales channel
channel_response = df.groupby(['sales_channel', 'response']).size().unstack(fill_value=0)
print("Response counts by channel:")
print(channel_response)
print()

In [ ]:
# Step 2: Calculate response rate (Yes / Total)
channel_response['total'] = channel_response['No'] + channel_response['Yes']
channel_response['response_rate'] = (channel_response['Yes'] / channel_response['total'] * 100).round(2)

print("Response rate by marketing channel:")
print(channel_response[['Yes', 'total', 'response_rate']].sort_values('response_rate', ascending=False))

In [ ]:
# Step 3: Use melt to convert to long format
# melt() unpivots a DataFrame from wide format to long format
# id_vars = columns to keep as-is
# value_vars = columns to unpivot
long_format = channel_response[['Yes', 'total', 'response_rate']].reset_index().melt(
    id_vars='sales_channel',
    value_vars=['Yes', 'total', 'response_rate'],
    var_name='metric',
    value_name='value'
)

print("Long format table:")
print(long_format)

**Insight**: The channel with the highest response rate is the most effective marketing channel. This should receive more budget allocation.